# PaySim

Same model pipeline as the IEEE-CIS notebooks, applied to the PaySim mobile-money dataset as a second, independent dataset. Runs its own dedicated Jaya hyperparameter search on PaySim (does not reuse the IEEE-CIS hyperparameters).

In [ ]:
# download PaySim dataset via kagglehub
import kagglehub

path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

In [ ]:
# load PaySim CSV and inspect
import pandas as pd
import numpy as np

csv_name = "PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(f"{path}/{csv_name}")

print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Fraud rate:")
print(df['isFraud'].value_counts())
print((df['isFraud'].value_counts(normalize=True) * 100).round(4))

In [ ]:
# print dataset summary
print("PAYSIM DATASET SUMMARY")
print(f"  Total transactions   : {len(df):,}")
print(f"  Legitimate (0)       : {(df['isFraud']==0).sum():,}")
print(f"  Fraud (1)            : {(df['isFraud']==1).sum():,}")
print(f"  Fraud rate           : {df['isFraud'].mean()*100:.4f}%")
print(f"  Number of features   : {df.shape[1]-1}")
print(f"  Missing values       : {df.isnull().sum().sum()}")
print(f"  Columns dropped      : 3 (nameOrig, nameDest, isFlaggedFraud — identifiers/rule-based flag)")
print("Preprocessing complete. Ready for SMOTENC.")

In [ ]:
# drop identifier / rule-based columns
df_clean = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

print("Shape after dropping identifiers:", df_clean.shape)
print("Columns:", df_clean.columns.tolist())

In [ ]:
# standard-scale numeric columns
from sklearn.preprocessing import StandardScaler

numeric_cols = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                 'oldbalanceDest', 'newbalanceDest']

scaler = StandardScaler()
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

print("Scaled columns:", numeric_cols)
print(df_clean[numeric_cols].describe().loc[['mean', 'std']].round(4))

In [ ]:
# carve out a stratified test set
from sklearn.model_selection import train_test_split

TEST_SIZE = 30000

df_test, df_remaining = train_test_split(
    df_clean,
    train_size=TEST_SIZE,
    stratify=df_clean['isFraud'],
    random_state=42
)

print("=== Test Set ===")
print(f"Total: {len(df_test):,}")
print(f"Fraud: {df_test['isFraud'].sum():,} ({df_test['isFraud'].mean()*100:.3f}%)")

In [ ]:
# build pre-SMOTE training pool (28K fraud / 42K legit target)
TARGET_FRAUD = 28000
TARGET_LEGIT = 42000

fraud_df = df_remaining[df_remaining['isFraud'] == 1]
legit_df = df_remaining[df_remaining['isFraud'] == 0]

print(f"Real fraud available (after test carve-out): {len(fraud_df):,}")
print(f"Real legit available: {len(legit_df):,}")

fraud_sample = fraud_df.copy()
legit_sample = legit_df.sample(n=TARGET_LEGIT, random_state=42)

df_train_pool = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nPre-SMOTE training pool:")
print(f"Total: {len(df_train_pool):,}")
print(f"Legit: {(df_train_pool['isFraud']==0).sum():,}")
print(f"Fraud (real): {(df_train_pool['isFraud']==1).sum():,}")
print(f"Synthetic fraud needed: {TARGET_FRAUD - len(fraud_sample):,}")

In [ ]:
# oversample minority class with SMOTENC
from imblearn.over_sampling import SMOTENC

X_train = df_train_pool.drop(columns=['isFraud'])
y_train = df_train_pool['isFraud']

categorical_cols = ['type']
categorical_indices = [X_train.columns.get_loc(c) for c in categorical_cols]

print(f"Before SMOTENC — Legit: {(y_train==0).sum():,} | Fraud: {(y_train==1).sum():,}")

smote_nc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy=TARGET_FRAUD / TARGET_LEGIT,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote_nc.fit_resample(X_train, y_train)

print(f"After SMOTENC  — Legit: {(y_train_smote==0).sum():,} | Fraud: {(y_train_smote==1).sum():,}")
print(f"Total: {len(y_train_smote):,} | Fraud %: {y_train_smote.mean()*100:.1f}")

In [ ]:
# one-hot encode transaction type
X_train_encoded = pd.get_dummies(X_train_smote, columns=['type'], dtype=float)

X_test = df_test.drop(columns=['isFraud'])
y_test = df_test['isFraud']
X_test_encoded = pd.get_dummies(X_test, columns=['type'], dtype=float)

X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Train shape:", X_train_encoded.shape)
print("Test shape: ", X_test_encoded.shape)

In [ ]:
# train/validation split (55K/15K)
from sklearn.model_selection import train_test_split

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_encoded, y_train_smote,
    test_size=15000,
    stratify=y_train_smote,
    random_state=42
)

X_test_final = X_test_encoded.values.astype(np.float32)
y_test_final = y_test.values
X_train_final = X_train_final.values.astype(np.float32)
X_val = X_val.values.astype(np.float32)
y_train_final = y_train_final.values
y_val = y_val.values

print("Train:", X_train_final.shape, "| Fraud:", (y_train_final==1).mean()*100, "%")
print("Val:  ", X_val.shape, "| Fraud:", (y_val==1).mean()*100, "%")
print("Test: ", X_test_final.shape, "| Fraud:", (y_test_final==1).mean()*100, "%")

In [ ]:
# logistic regression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_final, y_train_final)

lr_pred_val = lr_model.predict(X_val)
lr_prob_val = lr_model.predict_proba(X_val)[:, 1]
print("Logistic Regression — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, lr_pred_val)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, lr_pred_val, zero_division=0)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, lr_prob_val), 4))

In [ ]:
# XGBoost baseline
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', use_label_encoder=False)
xgb_model.fit(X_train_final, y_train_final)

xgb_pred_val = xgb_model.predict(X_val)
xgb_prob_val = xgb_model.predict_proba(X_val)[:, 1]
print("XGBoost — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, xgb_pred_val)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, xgb_pred_val, zero_division=0)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, xgb_prob_val), 4))

In [ ]:
# build + train ResNeXt-GRU
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

def build_rxt_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)
    path1 = layers.Dense(32, activation='relu')(x); path1 = layers.BatchNormalization()(path1)
    path2 = layers.Dense(32, activation='relu')(x); path2 = layers.BatchNormalization()(path2)
    path3 = layers.Dense(32, activation='relu')(x); path3 = layers.BatchNormalization()(path3)
    path4 = layers.Dense(32, activation='relu')(x); path4 = layers.BatchNormalization()(path4)
    merged = layers.concatenate([path1, path2, path3, path4])
    projected = layers.Dense(input_dim, activation='linear')(merged)
    rxt_out = layers.Activation('relu')(layers.add([x, projected]))
    gru_out = layers.GRU(128, dropout=0.3, return_sequences=False)(rxt_out)
    dense = layers.Dense(64, activation='relu')(gru_out)
    dropout = layers.Dropout(0.3)(dense)
    output = layers.Dense(1, activation='sigmoid')(dropout)
    model = models.Model(inputs=inputs, outputs=output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                           tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')])
    return model

tf.keras.backend.clear_session()
rxt_model = build_rxt_model(X_train_final.shape[1])

cw = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train_final)
cw = {0: cw[0], 1: cw[1]}

callbacks_rxt = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

print("Training ResNeXt-GRU on PaySim...")
history_rxt = rxt_model.fit(X_train_final, y_train_final, validation_data=(X_val, y_val),
                             epochs=50, batch_size=32, class_weight=cw, callbacks=callbacks_rxt, verbose=1)

rxt_prob_val = rxt_model.predict(X_val).flatten()
rxt_pred_val = (rxt_prob_val > 0.5).astype(int)
print("ResNeXt-GRU — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, rxt_pred_val)*100, 2), "%")
print("Precision:", round(precision_score(y_val, rxt_pred_val, zero_division=0)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val, rxt_pred_val, zero_division=0)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, rxt_pred_val, zero_division=0)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, rxt_prob_val), 4))

In [ ]:
# build + train ResNeXt-GRU + attention
def build_rxt_attention_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)
    p1 = layers.Dense(64, activation='relu')(x); p1 = layers.BatchNormalization()(p1)
    p2 = layers.Dense(64, activation='relu')(x); p2 = layers.BatchNormalization()(p2)
    p3 = layers.Dense(64, activation='relu')(x); p3 = layers.BatchNormalization()(p3)
    p4 = layers.Dense(64, activation='relu')(x); p4 = layers.BatchNormalization()(p4)
    merged1 = layers.concatenate([p1, p2, p3, p4])
    proj1 = layers.Dense(input_dim, activation='linear')(merged1)
    block1_out = layers.Activation('relu')(layers.add([x, proj1]))
    p5 = layers.Dense(64, activation='relu')(block1_out); p5 = layers.BatchNormalization()(p5)
    p6 = layers.Dense(64, activation='relu')(block1_out); p6 = layers.BatchNormalization()(p6)
    p7 = layers.Dense(64, activation='relu')(block1_out); p7 = layers.BatchNormalization()(p7)
    p8 = layers.Dense(64, activation='relu')(block1_out); p8 = layers.BatchNormalization()(p8)
    merged2 = layers.concatenate([p5, p6, p7, p8])
    proj2 = layers.Dense(input_dim, activation='linear')(merged2)
    block2_out = layers.Activation('relu')(layers.add([block1_out, proj2]))
    gru_out = layers.GRU(128, dropout=0.3, return_sequences=True)(block2_out)
    att_out = layers.MultiHeadAttention(num_heads=4, key_dim=32)(gru_out, gru_out)
    att_norm = layers.LayerNormalization()(layers.add([gru_out, att_out]))
    flat = layers.Flatten()(att_norm)
    dense1 = layers.Dense(128, activation='relu')(flat); drop1 = layers.Dropout(0.3)(dense1)
    dense2 = layers.Dense(64, activation='relu')(drop1); drop2 = layers.Dropout(0.2)(dense2)
    output = layers.Dense(1, activation='sigmoid')(drop2)
    model = models.Model(inputs=inputs, outputs=output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                           tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')])
    return model

tf.keras.backend.clear_session()
rxt_att_model = build_rxt_attention_model(X_train_final.shape[1])

print("Training ResNeXt-GRU + Attention on PaySim...")
history_att = rxt_att_model.fit(X_train_final, y_train_final, validation_data=(X_val, y_val),
                                 epochs=50, batch_size=32, class_weight=cw, callbacks=callbacks_rxt, verbose=1)

att_prob_val = rxt_att_model.predict(X_val).flatten()
att_pred_val = (att_prob_val > 0.5).astype(int)
print("ResNeXt-GRU + Attention — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, att_pred_val)*100, 2), "%")
print("Precision:", round(precision_score(y_val, att_pred_val, zero_division=0)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val, att_pred_val, zero_division=0)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, att_pred_val, zero_division=0)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, att_prob_val), 4))

In [ ]:
# Jaya hyperparameter search for RXT-J (population/generations search)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
import time
import gc
import pickle

POPULATION_SIZE = 6
GENERATIONS = 4
EPOCHS_PER_EVAL = 5
SEED = 42

BOUNDS = {
    'path_width':    (16, 64),
    'gru_units':      (64, 256),
    'dropout_rate':   (0.1, 0.5),
    'learning_rate':  (0.0001, 0.01),
    'weight_decay':   (0.00001, 0.001),
    'batch_size':     (16, 64),
}
DISCRETE_CHOICES = {
    'path_width': [16, 32, 64],
    'gru_units': [64, 128, 256],
    'batch_size': [16, 32, 64],
}

def snap_discrete(key, value):
    choices = DISCRETE_CHOICES[key]
    return int(min(choices, key=lambda c: abs(c - value)))

def random_candidate(rng):
    hp = {}
    for key, (lo, hi) in BOUNDS.items():
        val = rng.uniform(lo, hi)
        hp[key] = snap_discrete(key, val) if key in DISCRETE_CHOICES else round(val, 6)
    return hp

def clip_candidate(hp):
    clipped = {}
    for key, (lo, hi) in BOUNDS.items():
        val = min(max(hp[key], lo), hi)
        clipped[key] = snap_discrete(key, val) if key in DISCRETE_CHOICES else round(val, 6)
    return clipped

def build_rxt_j_model(input_dim, hp):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)
    paths = [layers.BatchNormalization()(layers.Dense(hp['path_width'], activation='relu')(x)) for _ in range(4)]
    merged = layers.concatenate(paths)
    projected = layers.Dense(input_dim, activation='linear')(merged)
    rxt_out = layers.Activation('relu')(layers.add([x, projected]))
    gru_out = layers.GRU(hp['gru_units'], dropout=hp['dropout_rate'], return_sequences=True)(rxt_out)
    att_out = layers.MultiHeadAttention(num_heads=4, key_dim=32)(gru_out, gru_out)
    att_norm = layers.LayerNormalization()(layers.add([gru_out, att_out]))
    flat = layers.Flatten()(att_norm)
    dense1 = layers.Dense(64, activation='relu')(flat)
    drop1 = layers.Dropout(hp['dropout_rate'])(dense1)
    output = layers.Dense(1, activation='sigmoid')(drop1)
    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=hp['learning_rate'], weight_decay=hp['weight_decay']),
        loss='binary_crossentropy', metrics=['accuracy']
    )
    return model

def objective(hp, tag=""):
    t0 = time.time()
    tf.keras.backend.clear_session()
    gc.collect()
    model = build_rxt_j_model(X_train_final.shape[1], hp)
    model.fit(
        X_train_final, y_train_final,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_PER_EVAL,
        batch_size=hp['batch_size'],
        verbose=0
    )
    prob = model.predict(X_val, verbose=0).flatten()
    pred = (prob > 0.5).astype(int)
    score = f1_score(y_val, pred, zero_division=0)
    del model, prob, pred
    gc.collect()
    tf.keras.backend.clear_session()
    print(f"  {tag} F1={score:.4f} ({time.time()-t0:.0f}s) — {hp}")
    return score

rng = np.random.default_rng(SEED)
population = [random_candidate(rng) for _ in range(POPULATION_SIZE)]

print("=== JAYA SEARCH ON PAYSIM ===")
print(f"Population={POPULATION_SIZE}, Generations={GENERATIONS}, Epochs/eval={EPOCHS_PER_EVAL}\n")

scores = [objective(hp, f"Gen0-{i}") for i, hp in enumerate(population)]
best_scores_by_gen = [max(scores)]

for gen in range(1, GENERATIONS + 1):
    print(f"\n--- Generation {gen} ---")
    best_idx = int(np.argmax(scores))
    worst_idx = int(np.argmin(scores))
    x_best = population[best_idx]
    x_worst = population[worst_idx]

    new_population, new_scores = [], []
    for i, (hp, score) in enumerate(zip(population, scores)):
        candidate = {}
        for key in BOUNDS:
            r1 = rng.uniform(0, 1)
            r2 = rng.uniform(0, 1)
            candidate[key] = hp[key] + r1 * (x_best[key] - abs(hp[key])) - r2 * (x_worst[key] - abs(hp[key]))
        candidate = clip_candidate(candidate)

        candidate_score = objective(candidate, f"Gen{gen}-{i}")
        if candidate_score > score:
            new_population.append(candidate); new_scores.append(candidate_score)
        else:
            new_population.append(hp); new_scores.append(score)

    population, scores = new_population, new_scores
    best_scores_by_gen.append(max(scores))
    print(f"Best F1 after generation {gen}: {max(scores):.4f}")

best_idx = int(np.argmax(scores))
best_hp_paysim = population[best_idx]

print(f"\n=== JAYA SEARCH COMPLETE (PaySim) ===")
print(f"Best hyperparameters found on PaySim: {best_hp_paysim}")
print(f"Best validation F1: {scores[best_idx]:.4f}")
print(f"F1 by generation: {[round(s, 4) for s in best_scores_by_gen]}")

with open("checkpoint_jaya_paysim_search.pkl", "wb") as f:
    pickle.dump({"best_hp": best_hp_paysim, "best_f1": scores[best_idx],
                 "f1_by_generation": best_scores_by_gen}, f)
print("Saved checkpoint_jaya_paysim_search.pkl")

In [ ]:
# train RXT-J + attention with fixed best hyperparameters
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

best_hp_paysim = {
    'path_width': 64,
    'gru_units': 256,
    'dropout_rate': 0.110846,
    'learning_rate': 0.003536,
    'weight_decay': 0.000504,
    'batch_size': 16
}

def build_rxt_j_attention_model(input_dim, hp):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)
    paths = [layers.BatchNormalization()(layers.Dense(hp['path_width'], activation='relu')(x))
             for _ in range(4)]
    merged = layers.concatenate(paths)
    projected = layers.Dense(input_dim, activation='linear')(merged)
    rxt_out = layers.Activation('relu')(layers.add([x, projected]))
    gru_out = layers.GRU(hp['gru_units'], dropout=hp['dropout_rate'], return_sequences=True)(rxt_out)
    att_out = layers.MultiHeadAttention(num_heads=4, key_dim=32)(gru_out, gru_out)
    att_norm = layers.LayerNormalization()(layers.add([gru_out, att_out]))
    flat = layers.Flatten()(att_norm)
    dense1 = layers.Dropout(hp['dropout_rate'])(layers.Dense(64, activation='relu')(flat))
    output = layers.Dense(1, activation='sigmoid')(dense1)
    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=hp['learning_rate'], weight_decay=hp['weight_decay']),
        loss='binary_crossentropy', metrics=['accuracy']
    )
    return model

rxt_j_attention_paysim = build_rxt_j_attention_model(X_train_final.shape[1], best_hp_paysim)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss')
]

pos_weight = (len(y_train_final) - sum(y_train_final)) / sum(y_train_final)

history_rxt_j_paysim = rxt_j_attention_paysim.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=best_hp_paysim['batch_size'],
    class_weight={0: 1.0, 1: pos_weight},
    callbacks=callbacks, verbose=1
)

rxt_j_attention_paysim.save("rxt_j_attention_paysim_model.keras")
print("Model saved to rxt_j_attention_paysim_model.keras")

p_paysim = rxt_j_attention_paysim.predict(X_test_final, verbose=0).flatten()
y_paysim = y_test_final
preds_default = (p_paysim > 0.5).astype(int)

print("\n" + "="*70)
print("RXT-J+Attention — Test Results (PaySim, default threshold 0.5)")
print("="*70)
print(f"Accuracy:  {accuracy_score(y_paysim, preds_default)*100:.2f}%")
print(f"Precision: {precision_score(y_paysim, preds_default, zero_division=0)*100:.2f}%")
print(f"Recall:    {recall_score(y_paysim, preds_default, zero_division=0)*100:.2f}%")
print(f"F1 Score:  {f1_score(y_paysim, preds_default, zero_division=0)*100:.2f}%")
print(f"AUC-ROC:   {roc_auc_score(y_paysim, p_paysim):.4f}")

print("\n" + "="*70)
print("THRESHOLD TUNING — RXT-J+Attention, PaySim")
print("="*70)
best_t, best_f1 = 0.5, 0
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    preds = (p_paysim > t).astype(int)
    acc  = accuracy_score(y_paysim, preds)
    prec = precision_score(y_paysim, preds, zero_division=0)
    rec  = recall_score(y_paysim, preds, zero_division=0)
    f1   = f1_score(y_paysim, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t
    print(f"  t={t}: acc={acc*100:.2f}% prec={prec*100:.2f}% rec={rec*100:.2f}% f1={f1*100:.2f}%")

In [ ]:
# evaluate all models on the PaySim test set
import pandas as pd

results_test_ps = {}
preds = {
    'Logistic Regression': (lr_model.predict(X_test_final), lr_model.predict_proba(X_test_final)[:,1]),
    'XGBoost':             (xgb_model.predict(X_test_final), xgb_model.predict_proba(X_test_final)[:,1]),
    'ResNeXt-GRU':          ((rxt_model.predict(X_test_final).flatten() > 0.5).astype(int), rxt_model.predict(X_test_final).flatten()),
    'ResNeXt-GRU + Attention': ((rxt_att_model.predict(X_test_final).flatten() > 0.5).astype(int), rxt_att_model.predict(X_test_final).flatten()),
    'RXT-J + Attention (Jaya)': ((rxt_j_attention_paysim.predict(X_test_final).flatten() > 0.5).astype(int), rxt_j_attention_paysim.predict(X_test_final).flatten()),
}

for name, (pred, prob) in preds.items():
    results_test_ps[name] = [
        accuracy_score(y_test_final, pred),
        precision_score(y_test_final, pred, zero_division=0),
        recall_score(y_test_final, pred, zero_division=0),
        f1_score(y_test_final, pred, zero_division=0),
        roc_auc_score(y_test_final, prob)
    ]

df_test_summary_paysim = pd.DataFrame.from_dict(results_test_ps, orient='index',
    columns=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'])
print("=== PAYSIM TEST SET SUMMARY (30K test) ===")
print(df_test_summary_paysim.round(4))

In [ ]:
# tune decision threshold per model
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

with open("checkpoint_paysim.pkl", "rb") as f:
    cp = pickle.load(f)

y_test = cp['y_test_final']
probs = cp['probs']

actual_fraud_pct = y_test.mean() * 100  

best_thresholds_paysim = {}

for model_name, y_prob in probs.items():
    print("="*70)
    print(f"THRESHOLD TUNING — {model_name} (PaySim, {actual_fraud_pct:.3f}% real fraud, {len(y_test):,} test)")
    print("="*70)
    print(f"{'Threshold':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'FalseAlarms':>12}")
    print("-"*65)

    best_t, best_f1 = 0.5, 0
    for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
        preds = (y_prob > t).astype(int)
        acc  = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec  = recall_score(y_test, preds, zero_division=0)
        f1   = f1_score(y_test, preds, zero_division=0)
        fa   = confusion_matrix(y_test, preds)[0][1]
        if f1 > best_f1:
            best_f1, best_t = f1, t
        mark = " <- best F1" if t == best_t and f1 == best_f1 else ""
        print(f"  {t:<10} {acc*100:>9.2f}% {prec*100:>9.2f}% {rec*100:>7.2f}% {f1*100:>7.2f}% {fa:>11,}{mark}")

    best_thresholds_paysim[model_name] = {'threshold': best_t, 'f1': best_f1}
    print(f"\nBest threshold: {best_t} | Best F1: {best_f1*100:.2f}%\n")

In [ ]:
# error analysis: false negatives / false positives by model
import pickle, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix

with open("checkpoint_paysim.pkl", "rb") as f:
    cp = pickle.load(f)

y_test_final = cp['y_test_final']
preds = cp['preds']

try:
    X_test_final
    feature_names_paysim = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                             'oldbalanceDest', 'newbalanceDest',
                             'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT',
                             'type_PAYMENT', 'type_TRANSFER']
    df_test = pd.DataFrame(X_test_final, columns=feature_names_paysim[:X_test_final.shape[1]])
    df_test['y_true'] = y_test_final
    have_features = True
except NameError:
    print("X_test_final not in memory — skipping feature-level breakdown, showing confusion matrix only.")
    have_features = False

models_to_analyze = {
    'RXT-J + Attention (Jaya)': preds['RXT-J + Attention (Jaya)'],
    'ResNeXt-GRU + Attention': preds['ResNeXt-GRU + Attention'],
}

for model_name, y_pred in models_to_analyze.items():
    print("="*70)
    print(f"ERROR ANALYSIS — {model_name} (PaySim, 0.16% real fraud, 30K test)")
    print("="*70)

    tn, fp, fn, tp = confusion_matrix(y_test_final, y_pred).ravel()
    print(f"True Positives (caught fraud):    {tp:>6,}")
    print(f"False Negatives (missed fraud):   {fn:>6,}  <- costliest errors")
    print(f"False Positives (false alarms):   {fp:>6,}")
    print(f"True Negatives (correct legit):   {tn:>6,}")
    print(f"Miss rate (FN / actual fraud):    {fn/(fn+tp)*100:.2f}%")
    print(f"False alarm rate (FP / actual legit): {fp/(fp+tn)*100:.2f}%")

    if have_features:
        df_test['pred'] = y_pred
        fn_rows = df_test[(df_test['y_true'] == 1) & (df_test['pred'] == 0)]
        fp_rows = df_test[(df_test['y_true'] == 0) & (df_test['pred'] == 1)]
        tp_rows = df_test[(df_test['y_true'] == 1) & (df_test['pred'] == 1)]

        print(f"\namount comparison:")
        print(f"  Missed fraud (FN) avg amount:  {fn_rows['amount'].mean():.2f}")
        print(f"  Caught fraud (TP) avg amount:  {tp_rows['amount'].mean():.2f}")
        print(f"  False alarms (FP) avg amount:  {fp_rows['amount'].mean():.2f}")

        if len(fn_rows) > 0:
            print(f"\nTop 5 highest-value MISSED frauds:")
            print(fn_rows.nlargest(min(5, len(fn_rows)), 'amount')[['amount']].to_string())

    print()

In [ ]:
# extract per-epoch training time from saved cell outputs
import json, re

def get_epoch_times(text):
    times = re.findall(r'\x1b\[1m(\d+)s\x1b\[0m', text)
    return [int(t) for t in times]

def get_output_text(cell):
    return ''.join(''.join(o.get('text', [])) for o in cell.get('outputs', []))

def find_model_cells(nb_path, keywords=('rxt_model.fit', 'rxt_att_model.fit', 'rxt_j_model')):
    nb = json.load(open(nb_path, encoding='utf-8'))
    print("="*20, nb_path, "="*20)
    for i, cell in enumerate(nb['cells']):
        src = ''.join(cell.get('source', []))
        for kw in keywords:
            if kw in src:
                print(i, '|', kw, '|', src.split(chr(10))[0][:70])
    print()

def extract_training_times(nb_path, model_cells, outlier_threshold=200):
    nb = json.load(open(nb_path, encoding='utf-8'))
    results = {}
    print("="*20, nb_path, "="*20)
    for idx, name in model_cells:
        text = get_output_text(nb['cells'][idx])
        times = get_epoch_times(text)
        raw_total = sum(times)
        clean_total = sum(t for t in times if t < outlier_threshold)
        n_outliers = sum(1 for t in times if t >= outlier_threshold)
        results[name] = {'epochs': len(times), 'raw_total_s': raw_total,
                          'clean_total_s': clean_total, 'outlier_epochs': n_outliers}
        print(f"{name:<28} epochs={len(times):>3} | raw={raw_total/60:>6.1f} min | "
              f"clean={clean_total/60:>6.1f} min | outliers={n_outliers}")
    print()
    return results

paysim_times = extract_training_times("paysim.ipynb", [
    (12, "ResNeXt-GRU"), (13, "ResNeXt-GRU + Attention"), (14, "RXT-J Jaya")
])